# NB00: Environment & Data Setup

**Goal**: Load datasets, build leakage-safe splits, save metadata. No model training.

### Mount Drive (Colab only)

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab — using local/XFAD_ROOT paths.')

def find_xfad_root() -> Path:
    '''Find repo root containing src/config.py (Colab Drive shortcuts included).'''
    candidates = []
    env = os.environ.get('XFAD_ROOT')
    if env:
        candidates.append(Path(env))
    candidates.extend([
        Path('/content/drive/MyDrive/XFAD'),
        Path.cwd(),
        Path.cwd().parent,
    ])
    for base in (Path('/content/drive/MyDrive'), Path('/content/drive/Shareddrives')):
        if not base.exists():
            continue
        for cfg in base.rglob('src/config.py'):
            try:
                if len(cfg.relative_to(base).parts) <= 6:
                    candidates.append(cfg.parent.parent)
            except ValueError:
                continue

    seen = set()
    for p in candidates:
        try:
            p = p.resolve()
        except OSError:
            continue
        if p in seen:
            continue
        seen.add(p)
        if (p / 'src' / 'config.py').exists():
            return p
    raise FileNotFoundError(
        "XFAD repo not found. After mounting Drive, set:\n"
        "  os.environ['XFAD_ROOT'] = '/content/drive/MyDrive/.../XFAD'\n"
        "then re-run this cell."
    )

REPO_ROOT = find_xfad_root()
os.environ['XFAD_ROOT'] = str(REPO_ROOT)
print('XFAD_ROOT:', REPO_ROOT)



## RUN MODE — edit the cell below (only switch in this notebook)

| `XFAD_DRY_RUN` | Meaning |
|---|---|
| `'1'` | **Smoke test** — tiny samples, fast sanity check (~minutes) |
| `'0'` | **Full paper run** — all data, downloads, full caches (~hours) |

Then: **Runtime → Run all** (restart session after changing the switch).


In [ ]:
import os

# ╔══════════════════════════════════════════════════════════════════════╗
# ║  ONLY SWITCH IN THIS NOTEBOOK                                        ║
# ║  '0' = FULL PAPER RUN (all data, both student arms)                  ║
# ║  '1' = SMOKE TEST (tiny data, baseline student only)                 ║
# ╚══════════════════════════════════════════════════════════════════════╝
os.environ['XFAD_DRY_RUN'] = '1'


In [ ]:
import os
import sys
from pathlib import Path

repo = Path(os.environ['XFAD_ROOT']) if os.environ.get('XFAD_ROOT') else None
if repo is None or not (repo / 'src' / 'config.py').exists():
    raise FileNotFoundError(
        "XFAD_ROOT not set. Run the Mount Drive cell first, or set "
        "os.environ['XFAD_ROOT'] to your XFAD folder."
    )
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
print('REPO_ROOT:', repo)

from src.utils.io import set_seed
set_seed(42)




In [ ]:
from src.utils.io import resolve_dry_run, print_run_banner
DRY_RUN = resolve_dry_run()
print_run_banner(notebook='NB00_Setup', dry_run=DRY_RUN)
# !pip install -r requirements.txt
import pandas as pd
try:
    import wfdb
except ImportError:
    import subprocess, sys
    print('Installing wfdb...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb', 'tqdm'])
    import wfdb
    print('wfdb installed')
from src.config import (
    MITBIH_EXCLUDED, MITBIH_DS1, MITBIH_DS1_TRAIN, MITBIH_DS1_VAL, MITBIH_DS2,
    PTBXL_TRAIN_FOLDS, PTBXL_VAL_FOLD, PTBXL_TEST_FOLD, PTBXL_NOISE_COLS,
    MITBIH_RAW, PTBXL_RAW, MITBIH_PROCESSED, PTBXL_PROCESSED, AAMI_MAP,
)
from src.utils.io import resolve_mitbih_raw_dir, ensure_dir
print('MITBIH_RAW target:', MITBIH_RAW)


### Download datasets (MIT-BIH + PTB-XL)

In [ ]:
from pathlib import Path
from src.data.download import download_all_datasets

# Full download: set XFAD_DRY_RUN=0 before this cell.
# Fast path (Colab): S3 -> /content/data/raw/ptbxl -> copy to Drive PTBXL_RAW.
# Set XFAD_DOWNLOAD_PTBXL_TEST=1 to also keep fold 10 (S3 sync already has all folds).
include_ptbxl_test = os.environ.get('XFAD_DOWNLOAD_PTBXL_TEST', '0') == '1'
ptbxl_folds = list(PTBXL_TRAIN_FOLDS) + [PTBXL_VAL_FOLD]
stage_dir = Path(os.environ.get('XFAD_PTBXL_STAGE', '/content/data/raw/ptbxl'))

download_stats = download_all_datasets(
    mitbih_dir=MITBIH_RAW,
    ptbxl_dir=PTBXL_RAW,
    mitbih_excluded=MITBIH_EXCLUDED,
    ptbxl_folds=ptbxl_folds,
    dry_run=DRY_RUN,
    include_ptbxl_test_fold=include_ptbxl_test,
    ptbxl_method=os.environ.get('XFAD_PTBXL_METHOD', 'auto'),  # auto|s3|wfdb
    ptbxl_stage_dir=stage_dir,
    dump_ptbxl_to_drive=os.environ.get('XFAD_DUMP_PTBXL_TO_DRIVE', '1') == '1',
)

print('MIT-BIH:', download_stats['mitbih'])
print('PTB-XL:', download_stats['ptbxl'])
print('MIT-BIH raw dir:', resolve_mitbih_raw_dir())
print('PTB-XL Drive dir:', PTBXL_RAW)
print('PTB-XL stage dir:', stage_dir)


**Fast Colab download (default):** PhysioNet S3 → `/content/data/raw/ptbxl` → copy to Drive.

**Storage layout**
- MIT-BIH (Drive): `data/raw/mitbih/*.dat`
- PTB-XL stage (Colab local): `/content/data/raw/ptbxl/`
- PTB-XL Drive dump: `data/raw/ptbxl/` (`ptbxl_database.csv` + `records500/`)

Interrupt any old per-file wfdb download first. Then set `XFAD_DRY_RUN=0` and re-run the download cell.
Env knobs: `XFAD_PTBXL_METHOD=auto|s3|wfdb`, `XFAD_DUMP_PTBXL_TO_DRIVE=1`, `XFAD_PTBXL_STAGE=/content/data/raw/ptbxl`.


In [ ]:
from pathlib import Path

mit_files = list(Path(resolve_mitbih_raw_dir()).glob('*.dat'))
ptb_csv = PTBXL_RAW / 'ptbxl_database.csv'
ptb_recs = list(PTBXL_RAW.glob('records500/**/*.dat')) if PTBXL_RAW.exists() else []
print(f'MIT-BIH .dat files: {len(mit_files)}')
print(f'PTB-XL CSV present: {ptb_csv.exists()}')
print(f'PTB-XL 500Hz .dat files: {len(ptb_recs)}')
if DRY_RUN:
    print('DRY_RUN=1 — no download performed. Set XFAD_DRY_RUN=0 for real data.')


### Build training caches (fast NB01 loading)


In [ ]:
# Verify this runtime imports the current src/ modules from Drive.
import importlib
import shutil
import sys
from pathlib import Path

REPO = Path(os.environ['XFAD_ROOT'])
REQUIRED_API = {
    'src.utils.io': ['resolve_dry_run', 'set_seed', 'ensure_dir', 'resolve_mitbih_raw_dir'],
    'src.data.loader_utils': ['make_loader'],
    'src.data.mitbih': ['MITBIHDataset', 'load_mitbih_samples'],
    'src.data.ptbxl': ['PTBXLDataset', 'load_ptbxl_samples'],
    'src.data.cache': [
        'build_mitbih_cache', 'build_ptbxl_cache', 'build_nb01_caches',
        'save_samples_npz', 'load_samples_npz', 'cache_has_groups', 'sync_processed_to_drive',
    ],
    'src.gate.calibration': ['fit_gate_model', 'calibrate_thresholds', 'risk_scores'],
    'src.gate.evaluation': ['collect_alarm_records', 'evaluate_baselines', 'records_to_tuples'],
    'src.metrics.significance': ['delong_roc_test', 'mcnemar_test', 'holm_bonferroni'],
}

# A leftover src/data/cache/ package directory would shadow src/data/cache.py.
stale = REPO / 'src' / 'data' / 'cache'
if stale.is_dir():
    shutil.rmtree(stale)
    print('Removed stale directory:', stale)

for name in [m for m in sys.modules if m == 'src' or m.startswith('src.')]:
    del sys.modules[name]

problems = []
for module_name, attributes in REQUIRED_API.items():
    try:
        module = importlib.import_module(module_name)
    except Exception as exc:  # noqa: BLE001 - report any import failure to the user
        problems.append(f'{module_name}: import failed ({exc})')
        continue
    missing = [a for a in attributes if not hasattr(module, a)]
    if missing:
        problems.append(f'{module_name}: missing {missing} -> {getattr(module, "__file__", "?")}')

if problems:
    print('\n'.join(problems))
    raise RuntimeError(
        'This runtime is using an out-of-date copy of src/. Let Google Drive finish syncing '
        '(or re-upload the repo), then Runtime > Restart session and rerun from the top.'
    )
print('src/ API verified at', REPO)


### Build split metadata

In [ ]:
print('MIT-BIH DS1_TRAIN', len(MITBIH_DS1_TRAIN), 'DS1_VAL', len(MITBIH_DS1_VAL), 'DS2', len(MITBIH_DS2))
# Beat-count sanity (when raw MIT-BIH present)
raw_dir = resolve_mitbih_raw_dir()
if any(raw_dir.glob('*.dat')) and not DRY_RUN:
    counts = {cls: 0 for cls in set(AAMI_MAP.values())}
    for rec in MITBIH_DS1:
        base = raw_dir / str(rec)
        if (base.with_suffix('.atr')).exists():
            ann = wfdb.rdann(str(base), 'atr')
            for sym in ann.symbol:
                if sym in AAMI_MAP:
                    counts[AAMI_MAP[sym]] += 1
    print('DS1 AAMI beat counts:', counts)

ptbxl_csv = PTBXL_RAW / 'ptbxl_database.csv'
ptbxl_train_ids, ptbxl_val_ids, ptbxl_test_ids = [], [], []
if ptbxl_csv.exists():
    df = pd.read_csv(ptbxl_csv, index_col='ecg_id')
    assert set(df['strat_fold'].dropna().unique()) == set(range(1, 11))
    ptbxl_train_ids = df[df['strat_fold'].isin(PTBXL_TRAIN_FOLDS)].index.tolist()
    ptbxl_val_ids = df[df['strat_fold'] == PTBXL_VAL_FOLD].index.tolist()
    ptbxl_test_ids = df[df['strat_fold'] == PTBXL_TEST_FOLD].index.tolist()
    noise_df = df[PTBXL_NOISE_COLS].copy()
    df = df.drop(columns=PTBXL_NOISE_COLS, errors='ignore')
    print('PTB-XL train/val/test:', len(ptbxl_train_ids), len(ptbxl_val_ids), len(ptbxl_test_ids))
else:
    noise_df = None
    print('PTB-XL CSV not found — metadata build skipped until data is present')


### Leakage tests (must pass)

In [ ]:
assert not set(MITBIH_DS1) & set(MITBIH_DS2)
assert not set(MITBIH_DS1_TRAIN) & set(MITBIH_DS1_VAL)
if ptbxl_train_ids:
    assert not set(ptbxl_train_ids) & set(ptbxl_val_ids)
    assert not set(ptbxl_train_ids) & set(ptbxl_test_ids)
    assert not set(ptbxl_val_ids) & set(ptbxl_test_ids)
print('Leakage tests passed')


### Save processed metadata

In [ ]:
ensure_dir(MITBIH_PROCESSED)
ensure_dir(PTBXL_PROCESSED)
mitbih_splits = pd.DataFrame(
    [{'record': r, 'split': 'train'} for r in MITBIH_DS1_TRAIN]
    + [{'record': r, 'split': 'val'} for r in MITBIH_DS1_VAL]
    + [{'record': r, 'split': 'test'} for r in MITBIH_DS2]
)
mitbih_splits.to_csv(MITBIH_PROCESSED / 'mitbih_splits.csv', index=False)
if ptbxl_csv.exists():
    df.to_csv(PTBXL_PROCESSED / 'ptbxl_processed.csv')
    noise_df.to_csv(PTBXL_PROCESSED / 'ptbxl_noise.csv')
print('Metadata saved')


In [ ]:
import os
import shutil
from pathlib import Path

# --- setup (run mount + XFAD_ROOT first) ---
repo = Path(os.environ['XFAD_ROOT'])
from src.config import MITBIH_PROCESSED, PTBXL_PROCESSED
from src.data.cache import sync_processed_to_drive

LOCAL_ROOT = Path(os.environ.get('XFAD_LOCAL_DATA', '/content/data')) / 'processed'

def copy_npz_to_drive(local_dir: Path, drive_dir: Path) -> list[str]:
    """Copy all .npz (+ manifest) from Colab local disk to Drive."""
    drive_dir.mkdir(parents=True, exist_ok=True)
    copied = []
    if not local_dir.exists():
        print(f'skip (missing): {local_dir}')
        return copied
    for path in local_dir.iterdir():
        if path.suffix in {'.npz', '.json'}:
            out = drive_dir / path.name
            shutil.copy2(path, out)
            copied.append(str(out))
            print(f'copied {path.name} -> {out}')
    return copied

# Option A: use built-in helper (recommended)
print('Built-in sync:', sync_processed_to_drive(dry_run=False))

# Option B: explicit copy (same result, easier to debug)
mit_copied = copy_npz_to_drive(LOCAL_ROOT / 'mitbih', MITBIH_PROCESSED)
ptb_copied = copy_npz_to_drive(LOCAL_ROOT / 'ptbxl', PTBXL_PROCESSED)

print('\nDrive MIT-BIH:', list(MITBIH_PROCESSED.glob('*.npz')))
print('Drive PTB-XL:', list(PTBXL_PROCESSED.glob('*.npz')))


In [ ]:
from src.data.cache import sync_processed_to_drive
print(sync_processed_to_drive(dry_run=False))

In [ ]:
import os
from pathlib import Path
from src.config import MITBIH_PROCESSED, PTBXL_PROCESSED
from src.data.cache import cache_has_groups

needed = [
    MITBIH_PROCESSED / 'ds1_train.npz',
    MITBIH_PROCESSED / 'ds1_val.npz',
    PTBXL_PROCESSED / 'train_folds_1-8.npz',
    PTBXL_PROCESSED / 'fold_9_val.npz',
]
for p in needed:
    size = f'({p.stat().st_size/1e6:.1f} MB)' if p.exists() else ''
    groups = 'groups=yes' if cache_has_groups(p) else 'groups=no'
    print(('OK  ' if p.exists() else 'MISS'), p, size, groups)
print(
    '\nCaches without group ids still train fine; NB05 rebuilds the test caches it needs '
    'with record/patient ids for patient-level confidence intervals.'
)
